# Making A Reward Model From NetHack Data

In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import AutoModel, AutoTokenizer
from torch.utils.data import Dataset, DataLoader
from tqdm import tqdm

import nle.dataset as nld
from nle.nethack import tty_render
from nle.env.tasks import NetHackChallenge
from nle.language_wrapper.wrappers import nle_language_wrapper as language_wrapper

CUDA_VISIBLE_DEVICES = 0
device = torch.device(f'cuda:{CUDA_VISIBLE_DEVICES}' if torch.cuda.is_available() else 'cpu')

/homes/53/fpinto/BALROG/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


# Dataset prep
Need to separate dataset into separate game trajectories, and return the state, action, current reward and reward-to-go (RTG) at each step

24x80 characters in one screen

In [3]:
dic = language_wrapper.NLELanguageWrapper.all_nle_action_map

actions = list(dic.keys())
act_str = list(dic.values())

for i in range(len(actions)):
	print(actions[i].name, actions[i].value, act_str[i])

HELP 63 ['help', '?']
PREVMSG 16 ['previous message', '^p']
N 107 ['north', 'k']
E 108 ['east', 'l']
S 106 ['south', 'j']
W 104 ['west', 'h']
NE 117 ['northeast', 'u']
SE 110 ['southeast', 'n']
SW 98 ['southwest', 'b']
NW 121 ['northwest', 'y']
N 75 ['far north', 'K']
E 76 ['far east', 'L']
S 74 ['far south', 'J']
W 72 ['far west', 'H']
NE 85 ['far northeast', 'U']
SE 78 ['far southeast', 'N']
SW 66 ['far southwest', 'B']
NW 89 ['far northwest', 'Y']
UP 60 ['up', '<']
DOWN 62 ['down', '>']
WAIT 46 ['wait', '.']
MORE 13 ['more', '\r', '\\r']
EXTCMD 35 ['extcmd', '#']
EXTLIST 191 ['extlist', 'M-?']
ADJUST 225 ['adjust', 'M-a']
ANNOTATE 193 ['annotate', 'M-A']
APPLY 97 ['apply', 'a']
ATTRIBUTES 24 ['attributes', '^x']
AUTOPICKUP 64 ['autopickup', '@']
CALL 67 ['call', 'C']
CAST 90 ['cast', 'Z']
CHAT 227 ['chat', 'M-c']
CLOSE 99 ['close', 'c']
CONDUCT 195 ['conduct', 'M-C']
DIP 228 ['dip', 'M-d']
DROP 100 ['drop', 'd']
DROPTYPE 68 ['droptype', 'D']
EAT 101 ['eat', 'e']
ESC 27 ['esc', '^[']

In [4]:
path_to_nld_aa_taster = "./nld-aa-taster/nle_data"
dbfilename = "ttyrecs.db" 
db_conn = nld.db.connect(filename=dbfilename)

dataset = nld.TtyrecDataset(
	"taster-dataset",
	batch_size=1,
	seq_length=128,
	dbfilename=dbfilename,
) # First batch will give timesteps 0-128 of batch_size games and the second batch will provide timesteps 32-63 for the same games, etc.
minibatch = next(iter(dataset))

In [6]:
print(minibatch.keys())
print([i for i in minibatch['keypresses']])

dict_keys(['tty_chars', 'tty_colors', 'tty_cursor', 'timestamps', 'done', 'gameids', 'keypresses', 'scores'])
[array([ 27,  27,  24,  32,  32, 229,  32,  64,  92,  32,  32,  58,  47,
        77,  32,  35, 116, 101,  13,  98,  27,  27,  27,  35, 116, 101,
        13,  98,  27,  92,  32,  32,  58,  47,  77,  32, 115, 121, 115,
        98, 115, 121, 115, 121, 115,  98, 115,  98, 115, 121, 115,  98,
       115, 104, 115, 104, 115,  68,  97,  13,  99,  13,  58,  32,  44,
        32,  58,  32,  44,  32,  44,  97,  13,  58, 121, 115, 104, 115,
       107, 115, 107, 115,  98, 115, 104, 115, 110, 115, 104, 115, 104,
       115,  98, 115, 107, 115, 107, 115, 106,  98, 115, 108, 117, 104,
        98, 104, 115, 104, 115, 121, 115, 104, 115, 104, 115,  35, 116,
       101,  13,  98,  27,  98, 115, 121, 115,  58,  44,  58], dtype=uint8)]


In [7]:
for i in minibatch['keypresses']:
	result = []
	for j in i:
		result.append(dic.get(j, []))
	print(result)

[['esc', '^['], ['esc', '^['], ['attributes', '^x'], ['space', ' '], ['space', ' '], ['enhance', 'M-e'], ['space', ' '], ['autopickup', '@'], ['known', '\\'], ['space', ' '], ['space', ' '], ['look', ':'], ['whatis', '/'], ['movefar', 'M'], ['space', ' '], ['extcmd', '#'], ['throw', 't'], ['eat', 'e'], ['more', '\r', '\\r'], ['southwest', 'b'], ['esc', '^['], ['esc', '^['], ['esc', '^['], ['extcmd', '#'], ['throw', 't'], ['eat', 'e'], ['more', '\r', '\\r'], ['southwest', 'b'], ['esc', '^['], ['known', '\\'], ['space', ' '], ['space', ' '], ['look', ':'], ['whatis', '/'], ['movefar', 'M'], ['space', ' '], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['northwest', 'y'], ['search', 's'], ['southwest', 'b'], ['search', 's'], ['west', 'h'], ['search', 's'], ['west', 'h'], ['search', 's'], ['droptype'

In [8]:
minibatch['scores'].shape

(1, 128)

In [9]:
print(minibatch['tty_chars'].shape)
print(minibatch['tty_colors'].shape)
print(minibatch['tty_cursor'].shape)

(1, 128, 24, 80)
(1, 128, 24, 80)
(1, 128, 2)


# Reward Model

In [10]:
class StateEncoder(nn.Module):
	"""Encodes the NetHack screen state into an embedding."""
	def __init__(self, embedding_dim=108): # NetHack screen is 12x9
		super().__init__()
		self.encoder = AutoModel.from_pretrained("all-MiniLM-L6-v2")
		self.projection = nn.Linear(self.encoder.config.hidden_size, embedding_dim)

	def forward(self, state_text):
		with torch.no_grad():
			encoded_state = self.encoder(state_text)["last_hidden_state"][:, 0, :]
		return self.projection(encoded_state)

In [11]:
class DecisionTransformer(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim, max_len=1920, pretrained_model="all-MiniLM-L6-v2"):
        super(DecisionTransformer, self).__init__()

        # Load the pre-trained MiniLM model
        self.state_encoder = AutoModel.from_pretrained(pretrained_model)
        self.tokenizer = AutoTokenizer.from_pretrained(pretrained_model)

        self.max_len = max_len
        self.embedding_dim = hidden_dim

        # Action embedding (each action gets embedded into hidden_dim space)
        self.action_embedding = nn.Embedding(action_dim, hidden_dim)

        # Transformer Encoder
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8), num_layers=6
        )

        # Output layer to predict the score (usefulness of the action)
        self.score_predictor = nn.Linear(hidden_dim, 1)  # Predict a scalar value (score)

        # Positional Encoding
        self.positional_encoding = nn.Parameter(torch.zeros(1, max_len, hidden_dim))

    def forward(self, states, actions, attention_mask=None):
        # 1. Tokenize and embed the grid (state) using the MiniLM encoder
        state_embeddings = self.state_encoder(states, attention_mask=attention_mask).last_hidden_state  # [batch_size, max_len, hidden_dim]

        # Ensure state_embeddings is (batch_size, max_len, state_dim)
        assert state_embeddings.size(1) == self.max_len, f"Expected max_len {self.max_len}, got {state_embeddings.size(1)}"
        assert state_embeddings.size(2) == self.state_encoder.config.hidden_size, f"Expected state_dim {self.state_encoder.config.hidden_size}, got {state_embeddings.size(2)}"
        
        # 2. Embed the actions (one-hot or integer encoded actions)
        action_embeddings = self.action_embedding(actions)  # [batch_size, k, hidden_dim] (k = number of top actions)

        # 3. Expand state embeddings for top k actions
        state_embeddings_expanded = state_embeddings.unsqueeze(1).expand(-1, actions.size(1), -1, -1)  # [batch_size, k, max_len, hidden_dim]
        
        # 4. Combine state and action embeddings
        x = torch.cat((state_embeddings_expanded, action_embeddings), dim=-1)  # [batch_size, k, max_len, hidden_dim*2]
        x += self.positional_encoding

        transformer_out = self.transformer(x.view(-1, x.size(2), x.size(3)), src_key_padding_mask=attention_mask)

        # Extract the relevant token's output (last token) for each action
        action_features = transformer_out[-1, :, :]  # [batch_size * k, hidden_dim]

        # Predict score for each action
        scores = self.score_predictor(action_features)  # [batch_size * k, 1]

        return scores.view(-1, actions.size(1))  # [batch_size, k]

In [12]:
def train(model, batch, optimizer, num_epochs=10):
	model.train()
	loss_fn = nn.CrossEntropyLoss()

	for epoch in tqdm(range(num_epochs)):
		total_loss = 0
		for i in batch:
			states, actions, rewards, rtgs = (
				batch["tt"].to(device),
				actions.to(device),
				rewards.to(device),
				rtgs.to(device),
			)

			optimizer.zero_grad()
			action_preds = model(states, actions, rewards, rtgs)
			loss = loss_fn(action_preds, actions)
			loss.backward()
			optimizer.step()

			total_loss += loss.item()

		print(f"Epoch {epoch+1}/{num_epochs} - Loss: {total_loss / len(batch):.4f}")

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim

class SimpleDecisionTransformer(nn.Module):
    def __init__(self, state_dim, action_dim, hidden_dim, max_len=128):
        super(SimpleDecisionTransformer, self).__init__()

        self.max_len = max_len
        self.embedding_dim = hidden_dim

        # Single state embedding (combined state from preprocess_state)
        self.state_embedding = nn.Embedding(state_dim, hidden_dim)  # state_dim corresponds to 24 * 80
        self.action_embedding = nn.Embedding(action_dim, hidden_dim)

        # Linear projection layer to match the embedding size after concatenation
        self.projection = nn.Linear(hidden_dim * 2, hidden_dim)  # Project concatenated state-action embeddings

        # Transformer Encoder
        self.transformer = nn.TransformerEncoder(
            nn.TransformerEncoderLayer(d_model=hidden_dim, nhead=8), num_layers=6
        )

        # Output layer to predict the score
        self.score_predictor = nn.Linear(hidden_dim, 1)  # Predicted score for each action

    def forward(self, state, action):
        # Embed state and action
        state_embedded = self.state_embedding(state)  # [batch_size, seq_len, hidden_dim]
        action_embedded = self.action_embedding(action)  # [batch_size, seq_len, hidden_dim]

        # Concatenate state and action embeddings
        x = torch.cat((state_embedded, action_embedded), dim=-1)  # [batch_size, seq_len, hidden_dim*2]

        # Project the concatenated embeddings to match the hidden_dim
        x = self.projection(x)  # [batch_size, seq_len, hidden_dim]

        # Pass through transformer
        transformer_out = self.transformer(x)  # [batch_size, seq_len, hidden_dim]

        # Use the last token's output to predict the score
        score_preds = self.score_predictor(transformer_out)  # [batch_size, seq_len, 1]

        return score_preds.squeeze(-1)  # [batch_size, seq_len] (scores for each action)


# Example training procedure

# Dummy data (for illustration)
batch_size = 2
seq_len = 5
state_dim = 24 * 80  # Flattened state (24x80 grid)
action_dim = 10  # Example action space size
hidden_dim = 64

# Example dummy data
states = torch.randint(0, state_dim, (batch_size, seq_len))  # Random state indices
actions = torch.randint(0, action_dim, (batch_size, seq_len))  # Random action indices
scores = torch.randn(batch_size, seq_len)  # Random scores (target)

# Initialize model
model = SimpleDecisionTransformer(state_dim, action_dim, hidden_dim)

# Loss and optimizer
criterion = nn.MSELoss()  # Mean Squared Error Loss for score prediction
optimizer = optim.Adam(model.parameters(), lr=1e-3)


model.train()
# Forward, loss and backprop
optimizer.zero_grad()
predicted_scores = model(states, actions)
loss = criterion(predicted_scores, scores)
loss.backward()
optimizer.step()

print(f"Predicted scores: {predicted_scores}")
print(f"Loss: {loss.item()}")

/homes/53/fpinto/BALROG/.venv/lib/python3.10/site-packages/torch/nn/modules/transformer.py:307: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.self_attn.batch_first was not True(use batch_first for better inference performance)
  warnings.warn(f"enable_nested_tensor is True, but self.use_nested_tensor is False because {why_not_sparsity_fast_path}")


Predicted scores: tensor([[-0.3193,  0.5727, -0.1933, -0.1989,  0.0324],
        [-0.4291,  0.2998,  0.3331, -0.2405,  0.0640]],
       grad_fn=<SqueezeBackward1>)
Loss: 0.7971262335777283
